# PyDBAdminKit 0.3.0 — Python API Lab

Notebook d’expérimentation de l’API Python publique de `pydbadminkit`.

Objectifs :
- valider la connexion et la configuration ;
- explorer les services `capability`, `server`, `catalog` et `security` ;
- manipuler les vrais objets du domaine ;
- tester le pipeline de mutation en **dry-run** ;
- garder les mutations réelles désactivées par défaut.

> Aucun mot de passe n’est stocké dans ce notebook. Définissez la variable d’environnement du profil avant de lancer Jupyter.


In [1]:
import os
from pathlib import Path
from uuid import uuid4

from pydbadminkit import __version__
from pydbadminkit.bootstrap import (
    build_capability_service,
    build_catalog_service,
    build_connection_service,
    build_security_mutation_service,
    build_security_service,
    build_server_service,
    resolve_connection,
)
from pydbadminkit.domain.common import DatabaseObjectType
from pydbadminkit.domain.safety import MutationOptions
from pydbadminkit.domain.security import CreateRoleCommand

In [ ]:
password_is_configured = bool(os.environ.get("PYDBADMIN_NATIVE_PASSWORD"))
print("PYDBADMIN_NATIVE_PASSWORD configured:", password_is_configured)
if not password_is_configured:
    print("Define PYDBADMIN_NATIVE_PASSWORD in your environment before connection tests.")


In [4]:
CONNECTION_PROFILE = "local-native"  # "local" pour Docker
RUN_MUTATIONS = (
    False  # passer explicitement à True uniquement sur une base de développement jetable
)


def find_config_path() -> Path:
    candidates = (
        Path.cwd() / "config.toml",
        Path.cwd().parent / "config.toml",
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("config.toml introuvable depuis le répertoire courant ou son parent")


CONFIG_PATH = find_config_path()
PASSWORD_ENV = (
    "PYDBADMIN_NATIVE_PASSWORD"
    if CONNECTION_PROFILE == "local-native"
    else "PYDBADMIN_LOCAL_PASSWORD"
)

if not os.environ.get(PASSWORD_ENV):
    raise RuntimeError(
        f"Définissez {PASSWORD_ENV} avant de démarrer Jupyter ; "
        "le secret ne doit pas être enregistré dans le notebook."
    )

print("Version :", __version__)
print("Profil  :", CONNECTION_PROFILE)
print("Config  :", CONFIG_PATH)
print("Secret  : défini via", PASSWORD_ENV)

Version : 0.5.0b1
Profil  : local-native
Config  : C:\Users\awounfouet\Projects\packages\pydbadminkit\config.toml
Secret  : défini via PYDBADMIN_NATIVE_PASSWORD


In [5]:
def section(title: str) -> None:
    print(f"\n{'=' * 72}\n{title}\n{'=' * 72}")


def show_object_ref(ref) -> str:
    return f"{ref.object_type.value}:{ref.name}"

## 1. Configuration résolue


In [6]:
config = resolve_connection(CONNECTION_PROFILE, CONFIG_PATH)
config

ResolvedConnectionConfig(name=ConnectionProfileName(value='local-native'), engine=<DatabaseEngine.POSTGRESQL: 'postgresql'>, host='localhost', port=5432, database='pydbadmin_dev', username='postgres', password=SecretValue(<redacted>), environment=<EnvironmentName.DEVELOPMENT: 'development'>, read_only=False, ssl=SSLConfig(mode=<SSLMode.DISABLE: 'disable'>, root_cert=None, cert=None, key=None), timeouts=TimeoutConfig(connect_seconds=10, statement_ms=None, lock_ms=None))

In [7]:
print("engine      :", config.engine)
print("host        :", config.host)
print("port        :", config.port)
print("database    :", config.database)
print("username    :", config.username)
print("environment :", config.environment)
print("read_only   :", config.read_only)
print("ssl_mode    :", config.ssl.mode)

engine      : postgresql
host        : localhost
port        : 5432
database    : pydbadmin_dev
username    : postgres
environment : development
read_only   : False
ssl_mode    : disable


## 2. Connection Service


In [9]:
connection_svc = build_connection_service(CONFIG_PATH)
connection_result = connection_svc.test(CONNECTION_PROFILE)
connection_result

ConnectionTestResult(engine=<DatabaseEngine.POSTGRESQL: 'postgresql'>, version=DatabaseVersion(major=17, minor=10, patch=0), current_database='pydbadmin_dev', current_user='postgres', latency_ms=256.4044000027934)

In [10]:
print("engine   :", connection_result.engine)
print("version  :", connection_result.version)
print("database :", connection_result.current_database)
print("user     :", connection_result.current_user)
print(f"latency  : {connection_result.latency_ms:.2f} ms")

engine   : postgresql
version  : 17.10
database : pydbadmin_dev
user     : postgres
latency  : 256.40 ms


## 3. Capabilities


In [11]:
capability_svc = build_capability_service()
capabilities = capability_svc.list()

for capability in capabilities:
    print(f"{capability.name:<38} {capability.availability.value}")

backup.create                          unavailable_tool
backup.restore                         unavailable_tool
backup.validate                        available
catalog.database.describe              available
catalog.database.list                  available
catalog.index.describe                 available
catalog.index.list                     available
catalog.schema.describe                available
catalog.schema.list                    available
catalog.table.describe                 available
catalog.table.list                     available
catalog.view.describe                  available
catalog.view.list                      available
connection.test                        available
maintenance.analyze                    available
maintenance.reindex                    available
maintenance.vacuum                     available
postgres.reindex.concurrently          available
postgres.reindex.progress              available
postgres.vacuum.progress               available
runtim

## 4. Server Service


In [12]:
server_svc = build_server_service(CONNECTION_PROFILE, CONFIG_PATH)
server_info = server_svc.get_info()
server_info

ServerInfo(engine=<DatabaseEngine.POSTGRESQL: 'postgresql'>, version=DatabaseVersion(major=17, minor=10, patch=0), current_database='pydbadmin_dev', current_user='postgres')

In [13]:
print("engine   :", server_info.engine)
print("version  :", server_info.version)
print("database :", server_info.current_database)
print("user     :", server_info.current_user)

engine   : postgresql
version  : 17.10
database : pydbadmin_dev
user     : postgres


## 5. Catalog Service — Object Explorer


In [14]:
catalog_svc = build_catalog_service(CONNECTION_PROFILE, CONFIG_PATH)

databases = catalog_svc.list_databases()
schemas = catalog_svc.list_schemas(include_system=False)

print("Databases:")
for database in databases:
    print(" -", database.name, "owner=", database.owner)

print("\nSchemas:")
for schema in schemas:
    print(" -", schema.name, "owner=", schema.owner)

Databases:
 - postgres owner= postgres
 - pydbadmin_dev owner= postgres

Schemas:
 - public owner= pg_database_owner


In [15]:
tables = catalog_svc.list_tables(schema="public")
print(f"{len(tables)} table(s) dans public")
for table in tables:
    print(" -", table.name, "owner=", table.owner, "kind=", table.kind.value)

if tables:
    description = catalog_svc.describe_table(tables[0].name)
    print("\nDescription de", description.table.name)
    for column in description.columns:
        print(" -", column.name, column.data_type, "nullable=", column.nullable)
    for constraint in description.constraints:
        print(" - constraint", constraint.constraint_type.value, constraint.name)

0 table(s) dans public


In [16]:
views = catalog_svc.list_views(schema="public")
indexes = catalog_svc.list_indexes(schema="public")

print("Views:")
for view in views:
    print(" -", view.name, "kind=", view.kind.value)

print("\nIndexes:")
for index in indexes:
    print(" -", index.name, "table=", index.table, "method=", index.method)

Views:

Indexes:


## 6. Security Service — lecture seule


In [17]:
security_svc = build_security_service(CONNECTION_PROFILE, CONFIG_PATH)
roles = security_svc.list_roles()

for role in roles:
    flags = []
    if role.can_login:
        flags.append("LOGIN")
    if role.is_superuser:
        flags.append("SUPERUSER")
    if role.can_create_db:
        flags.append("CREATEDB")
    print(f"{role.name:<24} {' '.join(flags)}")

postgres                 LOGIN SUPERUSER CREATEDB


In [18]:
postgres_description = security_svc.describe_role("postgres")
postgres_role = postgres_description.role

print("name        :", postgres_role.name)
print("login       :", postgres_role.can_login)
print("superuser   :", postgres_role.is_superuser)
print("createdb    :", postgres_role.can_create_db)
print("createrole  :", postgres_role.can_create_role)
print("inherit     :", postgres_role.inherit)
print("bypass_rls  :", postgres_role.bypass_rls)
print("member_of   :", [m.role for m in postgres_description.member_of])
print("members     :", [m.member for m in postgres_description.members])

name        : postgres
login       : True
superuser   : True
createdb    : True
createrole  : True
inherit     : True
bypass_rls  : True
member_of   : []
members     : []


In [19]:
direct_accesses = security_svc.list_direct_access("postgres")
for access in direct_accesses[:10]:
    print(
        show_object_ref(access.object),
        access.access_type.value,
        "issuer=",
        access.issuer,
        "delegable=",
        access.delegable,
    )

if not direct_accesses:
    print("Aucun accès direct explicite trouvé.")

Aucun accès direct explicite trouvé.


In [20]:
effective_accesses = security_svc.list_effective_access("postgres")
for access in effective_accesses[:10]:
    print(
        show_object_ref(access.object),
        access.access_type.value,
        "sources=",
        [source.value for source in access.sources],
    )

if not effective_accesses:
    print("Aucun accès effectif relationnel trouvé.")

Aucun accès effectif relationnel trouvé.


In [21]:
ownerships = security_svc.list_ownership(
    "postgres",
    object_type=DatabaseObjectType.TABLE,
)
for ownership in ownerships:
    print(show_object_ref(ownership.object), "owner=", ownership.owner)

if not ownerships:
    print("Aucune table possédée par postgres dans le périmètre inspecté.")

Aucune table possédée par postgres dans le périmètre inspecté.


## 7. Security Mutation Service

Le dry-run est toujours exécutable. Les mutations réelles restent désactivées tant que `RUN_MUTATIONS = False`. Le scénario réel crée un rôle temporaire unique puis le supprime dans un `finally`.


In [22]:
mutation_svc = build_security_mutation_service(CONNECTION_PROFILE, CONFIG_PATH)
TEMP_ROLE = f"demo_notebook_user_{uuid4().hex[:8]}"
create_command = CreateRoleCommand(name=TEMP_ROLE, can_login=True)
create_plan = mutation_svc.plan_create_role(create_command)

print("operation    :", create_plan.operation)
print("target       :", create_plan.target)
print("risk         :", create_plan.risk.label)
print("confirmation :", create_plan.confirmation.value)
print("correlation  :", create_plan.correlation_id)

dry_run = mutation_svc.create_role(
    create_command,
    MutationOptions(dry_run=True),
    plan=create_plan,
)
dry_run

operation    : security.role.create
target       : demo_notebook_user_1669cf1a
risk         : medium
confirmation : simple
correlation  : 0889d30e-7732-4db6-a8b9-fbfabe0f8a13


OperationPlan(operation='security.role.create', target='demo_notebook_user_1669cf1a', environment=<EnvironmentName.DEVELOPMENT: 'development'>, risk=<RiskLevel.MEDIUM: 20>, confirmation=<ConfirmationLevel.SIMPLE: 'simple'>, effects=("Create role 'demo_notebook_user_1669cf1a'.",), warnings=(), correlation_id='0889d30e-7732-4db6-a8b9-fbfabe0f8a13')

In [23]:
if not RUN_MUTATIONS:
    print("Mutations réelles désactivées. Passez RUN_MUTATIONS = True pour ce test.")
elif config.read_only or str(config.environment) not in {"development", "testing"}:
    print("Mutation refusée par le notebook : profil non development/testing ou read_only=true.")
else:
    created = False
    try:
        create_result = mutation_svc.create_role(
            create_command,
            MutationOptions(approved=True),
            plan=create_plan,
        )
        created = True
        print("CREATE:", create_result.status.value, create_result.message)
        print("ROLE  :", security_svc.describe_role(TEMP_ROLE).role)
    finally:
        if created:
            drop_plan = mutation_svc.plan_drop_role(TEMP_ROLE)
            drop_result = mutation_svc.drop_role(
                TEMP_ROLE,
                MutationOptions(approved=True),
                plan=drop_plan,
            )
            print("DROP  :", drop_result.status.value, drop_result.message)

Mutations réelles désactivées. Passez RUN_MUTATIONS = True pour ce test.


## Résultat attendu de ce lab

À la fin, vous devez avoir validé séparément : connexion, capabilities, server, catalog, security read-only et dry-run de mutation. Les cellules ne dépendent d’aucun état caché antérieur et les sorties ne sont pas versionnées.
